---

# Faz animação de Rastreamento de Flashes do GOES-16/19 para Intervalos de Tempo Definidos

---

- `OBJETIVO`:
> Plota animação de rastreamento de flashes.

- `DADOS DE ENTRADA`:
>  Tabela CSV contendo o tempo (tempo do primeiro evento do flash), latitude e longitude do flash. Exemplo de nome do arquivo: `flash_glm_goes_2020-06-30.csv`


- `DADOS DE SAÍDA`:
> 1. animação.gif

- `OBSERVAÇÕES`:
   > 1. Mudar os limites da imagem: lonmin, lonmax, latmin, latmax
   > 2. Muda a data: ano, mes, dia = '2020', '06', '30'

- `REALIZADO POR`:
> Enrique V. Mattos - 10/05/2026

- `ATUALIZADO POR`:
> Enrique V. Mattos - 10/05/2026
---

# **1° Passo:** Preparando ambiente

In [1]:
# instalações
!pip install -q ultraplot cartopy salem rasterio pyproj geopandas

# importa bibliotecas
import ultraplot as uplt
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
import pandas as pd
from datetime import timedelta, datetime
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import os
import imageio
import glob
import warnings
warnings.filterwarnings("ignore")

# monta o drive
from google.colab import drive
drive.mount('/content/drive')

# diretório raiz
dir = '/content/drive/MyDrive/2-PESQUISA/0_GLM/estudos_de_caso/2026-05-07e08_FENTRE_FRIA_RS'

# diretório de entrada
dir_input = f'{dir}/output/glm_diario_goes'

# diretório de saída
dir_output = f'{dir}/output/glm_figuras_goes/animacao'

# cria pasta de saída
os.makedirs(dir_output, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.5 MB/s eta 0:00:00
Mounted at /content/drive


# **Plota figura**

In [ ]:
%%time
#==================================================================================================#
#                                   DEFINIÇÃO DOS LIMITES DA IMAGEM
#==================================================================================================#
# limites das latitudes e longitudes
lonmin, lonmax, latmin, latmax = -67, -40, -43, -22

# extensão da imagem [min. lon, min. lat, max. lon, max. lat]
extent = [lonmin, latmin, lonmax, latmax]

#==================================================================================================#
#                                 LEITURA DO ARQUIVO NETCDF DIÁRIO
#==================================================================================================#
# leitura da planilha CSV
df_flash_07 = pd.read_csv(f'{dir_input}/flash_glm_goes_2026-05-07.csv')
df_flash_08 = pd.read_csv(f'{dir_input}/flash_glm_goes_2026-05-08.csv')

# juntar dataframes
df_flash = pd.concat([df_flash_07, df_flash_08])

# transforma a coluna data para índice do dataframe
df_flash.set_index('time', inplace=True)

# ordena o dataframe
df_flash.sort_index(inplace=True)

#==================================================================================================#
#                               DEFINE AS DATAS E INTERVALOS TEMPORAIS
#==================================================================================================#
# data INICIAL
anoi, mesi, diai, hori, mini = '2026', '05', '07', '00', '00'

# data FINAL
anof, mesf, diaf, horf, minf = '2026', '05', '08', '08', '00'

# frequência temporal do rastreamento [min]. Esta variável define de quanto em quanto tempo teremos os acumulados de flashes
frequencia_temporal = 60

# intervalo de acumulação dos flashes [min]
dt_acumulado = 60

# quantidade de tempos
ntimes = int((pd.to_datetime(f'{anof}{mesf}{diaf}{horf}{minf}') - pd.to_datetime(f'{anoi}{mesi}{diai}{hori}{mini}')).total_seconds() / 60 / frequencia_temporal) + 1

# extrai a data final. Exemplo: (anof, mesf, diaf, horf, minf) + 60min
data_final_str = f'{anof}-{mesf}-{diaf} {horf}:{minf}:00'
data_final = pd.to_datetime(data_final_str)
nova_data = data_final + timedelta(minutes=dt_acumulado)
ano_novo = nova_data.year
mes_novo = str(nova_data.month).zfill(2)
dia_novo = str(nova_data.day).zfill(2)
hor_novo = str(nova_data.hour).zfill(2)
min_novo = str(nova_data.minute).zfill(2)

#==================================================================================================#
#                                       DEFINE A PALETA DE CORES
#==================================================================================================#
# define a quantide de cores "cinza" e "jet"
n_cinza = int(ntimes * 0.3)  # 30% das cores em cinza
n_jet = ntimes - n_cinza     # 70% das cores em jet

# gera cores cinza (do mais claro ao mais escuro)
cores_cinza = plt.cm.Greys(np.linspace(0.3, 0.7, n_cinza))

# gera cores jet (espectro completo)
cores_jet = plt.cm.jet(np.linspace(0, 1, n_jet))

# combina as cores
cores_combinadas = np.vstack([cores_cinza, cores_jet])

# cria colormap personalizado
cmap_personalizada = LinearSegmentedColormap.from_list('cinza_jet', cores_combinadas)

# gera a paleta final de cores
cores = [cmap_personalizada(i/ntimes) for i in range(ntimes)]

#==================================================================================================#
#                                      LOOP DAS FIGURAS
#==================================================================================================#
# loop nos intervalos de tempos
for time, data in enumerate(pd.date_range(f'{anoi}{mesi}{diai}{hori}{mini}',f'{anof}{mesf}{diaf}{horf}{minf}', freq=f'{frequencia_temporal}min')):

    #----------------------------------------------------------------------#
    #                       INTERVALO DE TEMPO
    #----------------------------------------------------------------------#
    # intervalo INICIAL
    intervalo_1 = str(data)

    # intervalo FINAL
    intervalo_2 = str(data + timedelta(minutes=(dt_acumulado-1), seconds=59, microseconds=999998))

    # recorta o dado para o tempo atual
    df_horario = df_flash.loc[intervalo_1:intervalo_2]
    print('PROCESSANDO===>>>',intervalo_1 )

    #----------------------------------------------------------------------#
    #                       FORMATAÇÃO DO GRÁFICO
    #----------------------------------------------------------------------#
    # cria moldura da figura
    fig, ax = uplt.subplots(axheight=6.9, axwidth=6.8, tight=True, proj='pcarree')

    # formata os eixos
    ax.format(coast=True, borders=True, innerborders=False,
              labels=True, latlines=5, lonlines=5,
              latlim=(latmin, latmax), lonlim=(lonmin, lonmax),
              small='20px', large='25px',
              title=f'Rastreamento de Flashes \n',
              titleloc='l',
              titleweight='bold',
              titlecolor='bright red')

    #----------------------------------------------------------------------#
    #                       PLOTA FIGURA
    #----------------------------------------------------------------------#
    # cada figura usa uma cor diferente da paleta 'cores'
    cor_atual = cores[time]  # Obtém a cor correspondente ao índice atual

    ax.scatter(df_horario['lon'].values,
               df_horario['lat'].values,
               transform=ccrs.PlateCarree(),
               marker='o',
               s=6,
               color=cor_atual)  # Usa a cor específica para esta figura

    # plota contornos dos Estados
    shapefile = list(shpreader.Reader('https://github.com/evmpython/Minicurso_UFMS_SEMADESC_marco_2026/raw/main/01_utils/BR_UF_2019.shp').geometries())
    ax.add_geometries(shapefile, ccrs.PlateCarree(), edgecolor='gray', facecolor='none', linewidth=1.0)

    # contabiliza os flashes dentro da área plotada e dentro do período total
    df_filtered = df_horario[ (df_horario['lat'] >= latmin) & (df_horario['lat'] <= latmax) & (df_horario['lon'] >= lonmin) & (df_horario['lon'] <= lonmax)]

    # total de relâmpagos
    ax.text(lonmin+10., latmin+0.7, f'Total: {df_filtered.shape[0]} relâmpagos',
            color='red', fontsize=13, weight='bold',
            bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))

    # plota horário
    ax.text(lonmax-5.5, latmax-1, f'{intervalo_1[0:13]}h',
            color='red', fontsize=13, weight='bold',
            bbox=dict(facecolor='yellow', edgecolor='red', boxstyle='round,pad=0.3'))

    # plota subtítulo
    texto = (f'Satélite: GOES-19 (8km) | Frequencia Temporal: {frequencia_temporal}min | Acumulado: {dt_acumulado}min')
    ax.text(0.000, 1.043,
            texto,
            transform=ax.transAxes,
            color='gray',
            fontsize=8,
            verticalalignment='top')

    # salva figura
    fig.save(f'{dir_output}/GLM_05_flash_goes19_{intervalo_1[0:13]}_restemporal_{frequencia_temporal}min_dt_{dt_acumulado}min.jpg', bbox_inches='tight', dpi=300)

#==================================================================================================#
#                                         ANIMAÇÃO
#==================================================================================================#
# lista as imagens que serão usadas na animação
files = sorted(glob.glob(f'{dir_output}/*_restemporal_{frequencia_temporal}min_dt_{dt_acumulado}min.jpg'))

# faz animação
images = []
for file in files:
    images.append(imageio.imread(file))

# salva animação
imageio.mimsave(f'{dir_output}/animacao_restemporal_{frequencia_temporal}min_dt_{dt_acumulado}min.gif',
                images,
                duration=300,
                loop=0)

# mostra a animação
print("\nAbrindo o GIF..\n")
from IPython.display import Image
Image(open(f'{dir_output}/animacao_restemporal_{frequencia_temporal}min_dt_{dt_acumulado}min.gif','rb').read(), width=600)

PROCESSANDO===>>> 2026-05-07 00:00:00
PROCESSANDO===>>> 2026-05-07 01:00:00
PROCESSANDO===>>> 2026-05-07 02:00:00
PROCESSANDO===>>> 2026-05-07 03:00:00
PROCESSANDO===>>> 2026-05-07 04:00:00
PROCESSANDO===>>> 2026-05-07 05:00:00
PROCESSANDO===>>> 2026-05-07 06:00:00
PROCESSANDO===>>> 2026-05-07 07:00:00
PROCESSANDO===>>> 2026-05-07 08:00:00
PROCESSANDO===>>> 2026-05-07 09:00:00
PROCESSANDO===>>> 2026-05-07 10:00:00
PROCESSANDO===>>> 2026-05-07 11:00:00
PROCESSANDO===>>> 2026-05-07 12:00:00
PROCESSANDO===>>> 2026-05-07 13:00:00
